In [1]:
# ============================================
# CELL 1 ? IMPORTS + PATHS
# ============================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

STAGE0_DIR = Path("/kaggle/input/datasets/trinhdieulinh/stage0/stage0_artifacts")
STAGE4_INPUT_DIR = Path("/kaggle/input/datasets/trinhdieulinh/stage3-artifacts/stage3_artifacts")
STAGE4_DIR = Path("/kaggle/working/stage4_artifacts")
STAGE4_DIR.mkdir(parents=True, exist_ok=True)

COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"

print("Stage0 :", STAGE0_DIR)
print("Stage4 input :", STAGE4_INPUT_DIR)
print("Stage4 output:", STAGE4_DIR)


Stage0 : /kaggle/input/datasets/trinhdieulinh/stage0/stage0_artifacts
Stage4 input : /kaggle/input/datasets/trinhdieulinh/stage3-artifacts/stage3_artifacts
Stage4 output: /kaggle/working/stage4_artifacts


In [2]:
# ============================================
# CELL 2 ? LOAD STAGE 0 + BUILD TEST_WARM
# ============================================
with open(STAGE0_DIR / "split_meta.json", "r") as f:
    split_meta = json.load(f)

with open(STAGE0_DIR / "baseline_scores.json", "r") as f:
    baseline_scores = json.load(f)

train = pd.read_parquet(STAGE0_DIR / "train.parquet").copy()
test  = pd.read_parquet(STAGE0_DIR / "test.parquet").copy()
user_means_df = pd.read_parquet(STAGE0_DIR / "user_means.parquet")
item_means_df = pd.read_parquet(STAGE0_DIR / "item_means.parquet")

if COL_DATE in train.columns:
    train[COL_DATE] = pd.to_datetime(train[COL_DATE], errors="coerce")
if COL_DATE in test.columns:
    test[COL_DATE] = pd.to_datetime(test[COL_DATE], errors="coerce")

if COL_USER in user_means_df.columns:
    user_mean_map = dict(zip(user_means_df[COL_USER], user_means_df["user_mean"]))
else:
    user_mean_map = user_means_df["user_mean"].to_dict()

if COL_ITEM in item_means_df.columns:
    item_mean_map = dict(zip(item_means_df[COL_ITEM], item_means_df["item_mean"]))
else:
    item_mean_map = item_means_df["item_mean"].to_dict()

GLOBAL_MEAN = float(train[COL_RATING].mean())
train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())

test = test.copy()
test["user_known"] = test[COL_USER].isin(train_users)
test["item_known"] = test[COL_ITEM].isin(train_items)
mask_warm = test["user_known"] & test["item_known"]
test_warm = test.loc[mask_warm].reset_index(drop=True)

def clip_rating(x):
    return np.clip(np.asarray(x, dtype=np.float32), 1.0, 5.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def evaluate_predictions(y_true, y_pred, name="model"):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = clip_rating(y_pred)
    return {"model": name, "n": int(len(y_true)), "rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred)}

def baseline_predict(df_subset, user_map, item_map, global_mean):
    u = df_subset[COL_USER].map(user_map).fillna(global_mean)
    i = df_subset[COL_ITEM].map(item_map).fillna(global_mean)
    return ((u + i) / 2.0).clip(1.0, 5.0).values.astype(np.float32)

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

y_true = test_warm[COL_RATING].astype(np.float32).values
y_base = baseline_predict(test_warm, user_mean_map, item_mean_map, GLOBAL_MEAN)

print("Warm test size:", len(test_warm))
print(evaluate_predictions(y_true, y_base, "baseline"))
train_sorted_stage4 = train.sort_values(COL_DATE, na_position="last").reset_index(drop=True)
split_idx_stage4 = int(len(train_sorted_stage4) * 0.90)

train_fit_stage4 = train_sorted_stage4.iloc[:split_idx_stage4].copy().reset_index(drop=True)
val_fit_stage4 = train_sorted_stage4.iloc[split_idx_stage4:].copy().reset_index(drop=True)

print("Stage4 train_fit rows:", len(train_fit_stage4))
print("Stage4 val_fit rows  :", len(val_fit_stage4))

Warm test size: 240285
{'model': 'baseline', 'n': 240285, 'rmse': 1.102863868734165, 'mae': 0.8113042712211609}
Stage4 train_fit rows: 1018046
Stage4 val_fit rows  : 113117


In [3]:
# ============================================
# CELL 3 — LOAD STAGE 3 MODEL OUTPUTS
# ============================================
y_a = np.load(STAGE4_INPUT_DIR / "model_a_preds.npy")
y_b = np.load(STAGE4_INPUT_DIR / "model_b_preds.npy")
y_c = np.load(STAGE4_INPUT_DIR / "model_c_preds.npy")
y_d = np.load(STAGE4_INPUT_DIR / "model_d_preds.npy")

print("Loaded predictions:")
print("A:", y_a.shape)
print("B:", y_b.shape)
print("C:", y_c.shape)
print("D:", y_d.shape)

assert len(y_a) == len(test_warm)
assert len(y_b) == len(test_warm)
assert len(y_c) == len(test_warm)
assert len(y_d) == len(test_warm)

Loaded predictions:
A: (240285,)
B: (240285,)
C: (240285,)
D: (240285,)


In [4]:
# ============================================
# CELL 4 — QUICK SINGLE-MODEL CHECK
# ============================================
results_single = [
    evaluate_predictions(y_true, y_base, "baseline"),
    evaluate_predictions(y_true, y_a, "model_a"),
    evaluate_predictions(y_true, y_b, "model_b"),
    evaluate_predictions(y_true, y_c, "model_c"),
    evaluate_predictions(y_true, y_d, "model_d"),
]

results_single_df = pd.DataFrame(results_single).sort_values("rmse").reset_index(drop=True)
display(results_single_df)

,model,n,rmse,mae
0,baseline,240285,1.102864,0.811304
1,model_c,240285,1.102864,0.811304
2,model_d,240285,1.102868,0.811267
3,model_a,240285,1.105582,0.810892
4,model_b,240285,1.108873,0.764589


In [5]:
# ============================================
# CELL 5 ? ENSEMBLE HELPERS
# ============================================
import itertools
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

ADVANCED_FAMILIES = {"meta_ridge", "meta_elasticnet", "meta_hist_gradient_boosting", "group_aware_ridge"}


def weighted_blend(preds_dict, weights_dict):
    total_w = 0.0
    out = np.zeros_like(next(iter(preds_dict.values())), dtype=np.float32)
    for key, pred in preds_dict.items():
        w = float(weights_dict.get(key, 0.0))
        if w == 0.0:
            continue
        out += w * np.asarray(pred, dtype=np.float32)
        total_w += w
    if total_w <= 0:
        raise ValueError("All weights are zero.")
    return clip_rating(out / total_w)


def get_selection_mask(n_rows):
    """
    Use meta_select_mask if it already exists.
    Otherwise fallback to full OOF.
    """
    if "meta_select_mask" in globals():
        mask = np.asarray(meta_select_mask, dtype=bool)
        assert len(mask) == n_rows, "meta_select_mask length mismatch"
        return mask
    return np.ones(n_rows, dtype=bool)


def build_candidate_record(
    model_name,
    family,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    eligible_for_selection=True,
):
    weights = {} if weights is None else weights
    params = {} if params is None else params

    oof_pred = clip_rating(np.asarray(oof_pred, dtype=np.float32))
    test_pred = clip_rating(np.asarray(test_pred, dtype=np.float32))

    selection_mask = get_selection_mask(len(y_val_true))

    oof_select_res = evaluate_predictions(
        y_val_true[selection_mask],
        oof_pred[selection_mask],
        model_name,
    )

    oof_full_res = evaluate_predictions(
        y_val_true,
        oof_pred,
        model_name,
    )

    test_res = evaluate_predictions(y_true, test_pred, model_name)

    return {
        "model": model_name,
        "family": family,
        "eligible_for_selection": bool(eligible_for_selection),

        # dùng để sort/select
        "oof_rmse": float(oof_select_res["rmse"]),
        "oof_mae": float(oof_select_res["mae"]),

        # lưu thêm để debug
        "oof_full_rmse": float(oof_full_res["rmse"]),
        "oof_full_mae": float(oof_full_res["mae"]),

        "test_rmse": float(test_res["rmse"]),
        "test_mae": float(test_res["mae"]),
        "weights_json": json.dumps(to_jsonable(weights), sort_keys=True),
        "params_json": json.dumps(to_jsonable(params), sort_keys=True),
        "selection_source": "meta_select_from_oof_validation"
        if "meta_select_mask" in globals()
        else "full_oof_validation",
    }


def register_candidate(
    model_name,
    family,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    eligible_for_selection=True,
):
    record = build_candidate_record(
        model_name,
        family,
        oof_pred,
        test_pred,
        weights,
        params,
        eligible_for_selection,
    )

    compare_rows.append(record)

    candidate_store[model_name] = {
        "family": family,
        "oof_pred": clip_rating(np.asarray(oof_pred, dtype=np.float32)),
        "test_pred": clip_rating(np.asarray(test_pred, dtype=np.float32)),
        "weights": to_jsonable({} if weights is None else weights),
        "params": to_jsonable({} if params is None else params),
        "eligible_for_selection": bool(eligible_for_selection),
    }

    return record


def evaluate_per_star(y_true_arr, y_pred_arr):
    y_true_arr = np.asarray(y_true_arr, dtype=np.float32)
    y_pred_arr = clip_rating(y_pred_arr)
    star_values = np.rint(y_true_arr).astype(np.int32)
    out = {}
    for star in range(1, 6):
        mask = star_values == star
        out[f"star_{star}_n"] = int(mask.sum())
        if mask.sum() == 0:
            out[f"star_{star}_rmse"] = None
            out[f"star_{star}_mae"] = None
        else:
            out[f"star_{star}_rmse"] = rmse(y_true_arr[mask], y_pred_arr[mask])
            out[f"star_{star}_mae"] = mae(y_true_arr[mask], y_pred_arr[mask])
    return out


def simplex_weight_dicts(keys, step):
    total_units = int(round(1.0 / step))
    for units in itertools.product(range(total_units + 1), repeat=len(keys)):
        if sum(units) != total_units:
            continue
        yield {key: float(unit * step) for key, unit in zip(keys, units)}


def search_best_convex_blend(model_name, keys, step):
    best = None
    for weights in simplex_weight_dicts(keys, step):
        oof_pred = weighted_blend({key: pred_bank_val[key] for key in keys}, weights)
        test_pred = weighted_blend({key: pred_bank_test[key] for key in keys}, weights)
        record = build_candidate_record(model_name, "grid_blend", oof_pred, test_pred, weights, {"keys": list(keys), "step": float(step)}, eligible_for_selection=True)
        candidate = {
            "record": record,
            "oof_pred": oof_pred.copy(),
            "test_pred": test_pred.copy(),
            "weights": to_jsonable(weights),
            "params": {"keys": list(keys), "step": float(step)},
        }
        if best is None or (record["oof_rmse"], record["oof_mae"]) < (best["record"]["oof_rmse"], best["record"]["oof_mae"]):
            best = candidate
    return best


def search_best_residual_blend(step=0.10):
    base_val = pred_bank_val["base"]
    base_test = pred_bank_test["base"]
    best = None
    total_units = int(round(1.0 / step))
    for wb in range(total_units + 1):
        for wc in range(total_units + 1):
            for wd in range(total_units + 1):
                w_b = float(wb * step)
                w_c = float(wc * step)
                w_d = float(wd * step)
                oof_pred = clip_rating(base_val + w_b * (pred_bank_val["b"] - base_val) + w_c * (pred_bank_val["c"] - base_val) + w_d * (pred_bank_val["d"] - base_val))
                test_pred = clip_rating(base_test + w_b * (pred_bank_test["b"] - base_test) + w_c * (pred_bank_test["c"] - base_test) + w_d * (pred_bank_test["d"] - base_test))
                weights = {"base": 1.0, "w_b": w_b, "w_c": w_c, "w_d": w_d}
                params = {"step": float(step), "formula": "base + w_b*(B-base) + w_c*(C-base) + w_d*(D-base)"}
                record = build_candidate_record("residual_blend_base_b_c_d", "residual_blend", oof_pred, test_pred, weights, params, eligible_for_selection=True)
                candidate = {
                    "record": record,
                    "oof_pred": oof_pred.copy(),
                    "test_pred": test_pred.copy(),
                    "weights": to_jsonable(weights),
                    "params": params,
                }
                if best is None or (record["oof_rmse"], record["oof_mae"]) < (best["record"]["oof_rmse"], best["record"]["oof_mae"]):
                    best = candidate
    return best


def build_residual_meta_features(pred_base, pred_a, pred_b, pred_c, pred_d):
    pred_base = np.asarray(pred_base, dtype=np.float32)
    pred_a = np.asarray(pred_a, dtype=np.float32)
    pred_b = np.asarray(pred_b, dtype=np.float32)
    pred_c = np.asarray(pred_c, dtype=np.float32)
    pred_d = np.asarray(pred_d, dtype=np.float32)
    pred_stack = np.column_stack([pred_a, pred_b, pred_c, pred_d]).astype(np.float32)
    feature_names = [
        "b_minus_base",
        "c_minus_base",
        "d_minus_base",
        "a_minus_base",
        "abs_b_minus_base",
        "abs_c_minus_base",
        "abs_d_minus_base",
        "abs_a_minus_base",
        "prediction_mean",
        "prediction_std",
        "prediction_range",
        "base",
    ]
    X = np.column_stack([
        pred_b - pred_base,
        pred_c - pred_base,
        pred_d - pred_base,
        pred_a - pred_base,
        np.abs(pred_b - pred_base),
        np.abs(pred_c - pred_base),
        np.abs(pred_d - pred_base),
        np.abs(pred_a - pred_base),
        pred_stack.mean(axis=1),
        pred_stack.std(axis=1),
        pred_stack.max(axis=1) - pred_stack.min(axis=1),
        pred_base,
    ]).astype(np.float32)
    return X, feature_names


def build_group_labels(user_counts, item_counts):
    user_counts = np.asarray(user_counts, dtype=np.int32)
    item_counts = np.asarray(item_counts, dtype=np.int32)
    user_bin = np.where(user_counts <= 2, "low", np.where(user_counts <= 10, "mid", "high"))
    item_bin = np.where(item_counts <= 2, "low", np.where(item_counts <= 20, "mid", "high"))
    return np.char.add(np.char.add(user_bin.astype(str), "_"), item_bin.astype(str))


def fit_group_aware_ridge_candidate(X_val, X_test, y_val_target, fallback_val_pred, fallback_test_pred, val_groups, test_groups, min_rows=5000, alpha=1.0):
    X_val = np.asarray(X_val, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_val_target = np.asarray(y_val_target, dtype=np.float32)
    val_groups = np.asarray(val_groups)
    test_groups = np.asarray(test_groups)
    oof_pred = clip_rating(np.asarray(fallback_val_pred, dtype=np.float32).copy())
    test_pred = clip_rating(np.asarray(fallback_test_pred, dtype=np.float32).copy())
    group_summary = {}
    unique_groups = sorted(set(val_groups.tolist()) | set(test_groups.tolist()))
    for group in unique_groups:
        val_mask = val_groups == group
        test_mask = test_groups == group
        n_val = int(val_mask.sum())
        n_test = int(test_mask.sum())
        if n_val < min_rows:
            group_summary[group] = {"strategy": "fallback_global_ridge", "n_val": n_val, "n_test": n_test, "reason": f"n_val < {min_rows}"}
            continue
        try:
            model = Ridge(alpha=float(alpha), positive=True, fit_intercept=False)
            model.fit(X_val[val_mask], y_val_target[val_mask])
            oof_pred[val_mask] = clip_rating(model.predict(X_val[val_mask]))
            if n_test > 0:
                test_pred[test_mask] = clip_rating(model.predict(X_test[test_mask]))
            coef = model.coef_.astype(np.float32)
            group_summary[group] = {
                "strategy": "group_specific_positive_ridge",
                "n_val": n_val,
                "n_test": n_test,
                "alpha": float(alpha),
                "coef": {"base": float(coef[0]), "b": float(coef[1]), "c": float(coef[2]), "d": float(coef[3])},
            }
        except Exception as exc:
            group_summary[group] = {"strategy": "fallback_global_ridge", "n_val": n_val, "n_test": n_test, "reason": str(exc)}
    return clip_rating(oof_pred), clip_rating(test_pred), group_summary
def make_meta_split(n_rows, fit_ratio=0.70):
    split_idx = int(n_rows * fit_ratio)
    meta_fit_mask = np.zeros(n_rows, dtype=bool)
    meta_select_mask = np.zeros(n_rows, dtype=bool)
    meta_fit_mask[:split_idx] = True
    meta_select_mask[split_idx:] = True
    return meta_fit_mask, meta_select_mask



In [6]:
# ============================================
# CELL 6 ? LOAD STRICT OOF VALIDATION INPUTS
# ============================================
y_val_true = np.load(STAGE4_INPUT_DIR / "y_val_true_oof.npy")
y_val_base = np.load(STAGE4_INPUT_DIR / "baseline_val_preds_oof.npy")
y_val_a    = np.load(STAGE4_INPUT_DIR / "model_a_val_preds_oof.npy")
y_val_b    = np.load(STAGE4_INPUT_DIR / "model_b_val_preds_oof.npy")
y_val_c    = np.load(STAGE4_INPUT_DIR / "model_c_val_preds_oof.npy")
y_val_d    = np.load(STAGE4_INPUT_DIR / "model_d_val_preds_oof.npy")
validation_index_oof = pd.read_csv(STAGE4_INPUT_DIR / "validation_index_oof.csv")

for required_col in [COL_USER, COL_ITEM]:
    assert required_col in validation_index_oof.columns, f"Missing column in validation_index_oof.csv: {required_col}"

for arr_name, arr in {"baseline": y_val_base, "model_a": y_val_a, "model_b": y_val_b, "model_c": y_val_c, "model_d": y_val_d}.items():
    assert len(arr) == len(y_val_true), f"Length mismatch for {arr_name}: {len(arr)} vs {len(y_val_true)}"
assert len(validation_index_oof) == len(y_val_true), f"validation_index_oof length mismatch: {len(validation_index_oof)} vs {len(y_val_true)}"
for arr_name, arr in {"baseline": y_base, "model_a": y_a, "model_b": y_b, "model_c": y_c, "model_d": y_d}.items():
    assert len(arr) == len(y_true), f"Test prediction length mismatch for {arr_name}: {len(arr)} vs {len(y_true)}"

pred_bank_val = {"base": y_val_base, "a": y_val_a, "b": y_val_b, "c": y_val_c, "d": y_val_d}
pred_bank_test = {"base": y_base, "a": y_a, "b": y_b, "c": y_c, "d": y_d}

compare_rows = []
candidate_store = {}
advanced_diagnostics = {"candidate_search": {}, "skipped_candidates": {}, "meta_feature_names": []}

reference_candidates = {
    "baseline": ("single_model", y_val_base, y_base, {"base": 1.0}, {"kind": "single_model"}),
    "model_a": ("single_model", y_val_a, y_a, {"a": 1.0}, {"kind": "single_model"}),
    "model_b": ("single_model", y_val_b, y_b, {"b": 1.0}, {"kind": "single_model"}),
    "model_c": ("single_model", y_val_c, y_c, {"c": 1.0}, {"kind": "single_model"}),
    "model_d": ("single_model", y_val_d, y_d, {"d": 1.0}, {"kind": "single_model"}),
}
for name, (family, oof_pred, test_pred, weights, params) in reference_candidates.items():
    register_candidate(name, family, oof_pred, test_pred, weights=weights, params=params, eligible_for_selection=True)

reference_df = pd.DataFrame(compare_rows).sort_values(["oof_rmse", "oof_mae", "model"]).reset_index(drop=True)
display(reference_df)
print("Loaded strict OOF rows:", len(y_val_true))
print("Loaded validation_index_oof rows:", len(validation_index_oof))
meta_fit_mask, meta_select_mask = make_meta_split(len(y_val_true), fit_ratio=0.70)
print("Meta fit rows   :", int(meta_fit_mask.sum()))
print("Meta select rows:", int(meta_select_mask.sum()))

,model,family,eligible_for_selection,oof_rmse,oof_mae,oof_full_rmse,oof_full_mae,test_rmse,test_mae,weights_json,params_json,selection_source
0,model_d,single_model,True,1.046053,0.743093,1.046053,0.743093,1.102868,0.811267,"{""d"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
1,baseline,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""base"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
2,model_c,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""c"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
3,model_b,single_model,True,1.046745,0.726074,1.046745,0.726074,1.108873,0.764589,"{""b"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
4,model_a,single_model,True,1.048409,0.742963,1.048409,0.742963,1.105582,0.810892,"{""a"": 1.0}","{""kind"": ""single_model""}",full_oof_validation


Loaded strict OOF rows: 113117
Loaded validation_index_oof rows: 113117
Meta fit rows   : 79181
Meta select rows: 33936


In [7]:
# ============================================
# CELL 7 — MANUAL BLENDS + GRID BLENDS ON META_SELECT
# ============================================

manual_configs = {
    "blend_base_b_manual": {
        "base": 0.50,
        "b": 0.50,
    },
    "blend_base_b_c_manual": {
        "base": 0.55,
        "b": 0.35,
        "c": 0.10,
    },
    "blend_base_b_c_d_manual": {
        "base": 0.55,
        "b": 0.30,
        "c": 0.10,
        "d": 0.05,
    },
}

for name, weights in manual_configs.items():
    oof_pred = weighted_blend(pred_bank_val, weights)
    test_pred = weighted_blend(pred_bank_test, weights)

    register_candidate(
        name,
        "manual_blend",
        oof_pred,
        test_pred,
        weights=weights,
        params={
            "kind": "manual",
            "selection_source": "meta_select_from_oof_validation",
        },
        eligible_for_selection=True,
    )


grid_base_b = search_best_convex_blend(
    "grid_base_b",
    ("base", "b"),
    step=0.05,
)

register_candidate(
    "grid_base_b",
    "grid_blend",
    grid_base_b["oof_pred"],
    grid_base_b["test_pred"],
    weights=grid_base_b["weights"],
    params={
        **grid_base_b["params"],
        "selection_source": "meta_select_from_oof_validation",
    },
    eligible_for_selection=True,
)


grid_base_b_c = search_best_convex_blend(
    "grid_base_b_c",
    ("base", "b", "c"),
    step=0.05,
)

register_candidate(
    "grid_base_b_c",
    "grid_blend",
    grid_base_b_c["oof_pred"],
    grid_base_b_c["test_pred"],
    weights=grid_base_b_c["weights"],
    params={
        **grid_base_b_c["params"],
        "selection_source": "meta_select_from_oof_validation",
    },
    eligible_for_selection=True,
)


grid_base_b_c_d = search_best_convex_blend(
    "grid_base_b_c_d",
    ("base", "b", "c", "d"),
    step=0.10,
)

register_candidate(
    "grid_base_b_c_d",
    "grid_blend",
    grid_base_b_c_d["oof_pred"],
    grid_base_b_c_d["test_pred"],
    weights=grid_base_b_c_d["weights"],
    params={
        **grid_base_b_c_d["params"],
        "selection_source": "meta_select_from_oof_validation",
    },
    eligible_for_selection=True,
)


blend_preview_df = (
    pd.DataFrame(compare_rows)
    .sort_values(
        ["eligible_for_selection", "oof_rmse", "oof_mae", "model"],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

display(blend_preview_df)

,model,family,eligible_for_selection,oof_rmse,oof_mae,oof_full_rmse,oof_full_mae,test_rmse,test_mae,weights_json,params_json,selection_source
0,model_d,single_model,True,1.046053,0.743093,1.046053,0.743093,1.102868,0.811267,"{""d"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
1,baseline,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""base"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
2,model_c,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""c"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
3,model_b,single_model,True,1.046745,0.726074,1.046745,0.726074,1.108873,0.764589,"{""b"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
4,model_a,single_model,True,1.048409,0.742963,1.048409,0.742963,1.105582,0.810892,"{""a"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
5,grid_base_b_c,grid_blend,True,1.065895,0.746736,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.15000000000000002, ""c"": 0...","{""keys"": [""base"", ""b"", ""c""], ""selection_source...",meta_select_from_oof_validation
6,grid_base_b_c_d,grid_blend,True,1.065895,0.746736,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.2, ""c"": 0.300000000000000...","{""keys"": [""base"", ""b"", ""c"", ""d""], ""selection_s...",meta_select_from_oof_validation
7,blend_base_b_manual,manual_blend,True,1.065895,0.746737,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.5}","{""kind"": ""manual"", ""selection_source"": ""meta_s...",meta_select_from_oof_validation
8,grid_base_b,grid_blend,True,1.065895,0.746737,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.5}","{""keys"": [""base"", ""b""], ""selection_source"": ""m...",meta_select_from_oof_validation
9,blend_base_b_c_manual,manual_blend,True,1.065978,0.749327,1.045231,0.736903,1.094171,0.791539,"{""b"": 0.35, ""base"": 0.55, ""c"": 0.1}","{""kind"": ""manual"", ""selection_source"": ""meta_s...",meta_select_from_oof_validation


In [8]:
# ============================================
# CELL 8 — RESIDUAL BLEND + GLOBAL RIDGE + ADVANCED META
# ============================================

print("=" * 70)
print("CELL 8 — RESIDUAL BLEND + GLOBAL RIDGE + ADVANCED META")
print("=" * 70)

# -------------------------------------------------
# 8.1 Residual blend
# -------------------------------------------------
best_residual = search_best_residual_blend(step=0.10)

register_candidate(
    "residual_blend_base_b_c_d",
    "residual_blend",
    best_residual["oof_pred"],
    best_residual["test_pred"],
    weights=best_residual["weights"],
    params=best_residual["params"],
    eligible_for_selection=True,
)

advanced_diagnostics["candidate_search"]["residual_blend_base_b_c_d"] = {
    "selected": best_residual["params"],
    "weights": best_residual["weights"],
}


# -------------------------------------------------
# 8.2 Global Ridge positive: base + B + C + D
# -------------------------------------------------
X_val_base_b_c_d = np.column_stack([
    y_val_base,
    y_val_b,
    y_val_c,
    y_val_d,
]).astype(np.float32)

X_test_base_b_c_d = np.column_stack([
    y_base,
    y_b,
    y_c,
    y_d,
]).astype(np.float32)

ridge = Ridge(
    alpha=1.0,
    positive=True,
    fit_intercept=False,
)

ridge.fit(
    X_val_base_b_c_d[meta_fit_mask],
    y_val_true[meta_fit_mask],
)

y_pred_ridge_oof = clip_rating(
    ridge.predict(X_val_base_b_c_d)
)

y_pred_ridge = clip_rating(
    ridge.predict(X_test_base_b_c_d)
)

ridge_coef = ridge.coef_.astype(np.float32)
ridge_coef_sum = float(ridge_coef.sum())
ridge_coef_norm = (
    (ridge_coef / ridge_coef_sum).astype(np.float32)
    if ridge_coef_sum > 0
    else ridge_coef.copy()
)

ridge_weights = {
    "base": float(ridge_coef[0]),
    "b": float(ridge_coef[1]),
    "c": float(ridge_coef[2]),
    "d": float(ridge_coef[3]),
    "normalized_base": float(ridge_coef_norm[0]),
    "normalized_b": float(ridge_coef_norm[1]),
    "normalized_c": float(ridge_coef_norm[2]),
    "normalized_d": float(ridge_coef_norm[3]),
}

ridge_params = {
    "alpha": 1.0,
    "positive": True,
    "fit_intercept": False,
    "features": ["base", "b", "c", "d"],
    "selection_split": "meta_fit_70_meta_select_30",
}

register_candidate(
    "ridge_positive_base_b_c_d",
    "ridge_positive",
    y_pred_ridge_oof,
    y_pred_ridge,
    weights=ridge_weights,
    params=ridge_params,
    eligible_for_selection=True,
)

advanced_diagnostics["candidate_search"]["ridge_positive_base_b_c_d"] = {
    "params": ridge_params,
    "weights": ridge_weights,
}


# -------------------------------------------------
# 8.3 Build residual meta features
# -------------------------------------------------
X_val_meta, meta_feature_names = build_residual_meta_features(
    y_val_base,
    y_val_a,
    y_val_b,
    y_val_c,
    y_val_d,
)

X_test_meta, _ = build_residual_meta_features(
    y_base,
    y_a,
    y_b,
    y_c,
    y_d,
)

y_res_val = (y_val_true - y_val_base).astype(np.float32)

advanced_diagnostics["meta_feature_names"] = meta_feature_names


# -------------------------------------------------
# 8.4 Advanced 1 — Residual Ridge Meta
# -------------------------------------------------
ridge_meta_rows = []
best_ridge_meta_cfg = None
best_ridge_meta_key = (float("inf"), float("inf"))

for alpha in [0.01, 0.1, 1.0, 5.0, 10.0]:
    model = Ridge(
        alpha=float(alpha),
        fit_intercept=True,
    )

    model.fit(
        X_val_meta[meta_fit_mask],
        y_res_val[meta_fit_mask],
    )

    oof_pred_full = clip_rating(
        y_val_base + model.predict(X_val_meta)
    )

    oof_select_res = evaluate_predictions(
        y_val_true[meta_select_mask],
        oof_pred_full[meta_select_mask],
        f"residual_ridge_meta_alpha_{alpha}",
    )

    row = {
        "alpha": float(alpha),
        "meta_select_rmse": float(oof_select_res["rmse"]),
        "meta_select_mae": float(oof_select_res["mae"]),
    }
    ridge_meta_rows.append(row)

    key = (
        float(oof_select_res["rmse"]),
        float(oof_select_res["mae"]),
    )

    if key < best_ridge_meta_key:
        best_ridge_meta_key = key
        best_ridge_meta_cfg = {
            "alpha": float(alpha),
        }

if best_ridge_meta_cfg is not None:
    best_alpha = float(best_ridge_meta_cfg["alpha"])

    ridge_meta_final = Ridge(
        alpha=best_alpha,
        fit_intercept=True,
    )

    # refit selected config on full OOF for final test prediction
    ridge_meta_final.fit(
        X_val_meta,
        y_res_val,
    )

    ridge_meta_oof_pred = clip_rating(
        y_val_base + ridge_meta_final.predict(X_val_meta)
    )

    ridge_meta_test_pred = clip_rating(
        y_base + ridge_meta_final.predict(X_test_meta)
    )

    ridge_meta_weights = {
        "intercept": float(ridge_meta_final.intercept_),
        "coef_by_feature": {
            name: float(coef)
            for name, coef in zip(meta_feature_names, ridge_meta_final.coef_)
        },
    }

    ridge_meta_params = {
        "alpha": float(best_alpha),
        "fit_intercept": True,
        "feature_names": meta_feature_names,
        "selection_split": "meta_fit_70_meta_select_30",
    }

    register_candidate(
        "residual_ridge_meta",
        "meta_ridge",
        ridge_meta_oof_pred,
        ridge_meta_test_pred,
        weights=ridge_meta_weights,
        params=ridge_meta_params,
        eligible_for_selection=True,
    )

    advanced_diagnostics["candidate_search"]["residual_ridge_meta"] = {
        "grid_results": ridge_meta_rows,
        "selected": ridge_meta_params,
        "weights": ridge_meta_weights,
    }

    print("Selected residual_ridge_meta:", ridge_meta_params)


# -------------------------------------------------
# 8.5 Advanced 2 — ElasticNet Residual Meta
# -------------------------------------------------
elasticnet_rows = []
best_elasticnet_cfg = None
best_elasticnet_key = (float("inf"), float("inf"))

try:
    for alpha in [0.0001, 0.001, 0.01]:
        for l1_ratio in [0.1, 0.5]:
            model = ElasticNet(
                alpha=float(alpha),
                l1_ratio=float(l1_ratio),
                fit_intercept=True,
                max_iter=3000,
                tol=1e-3,
                random_state=42,
            )

            model.fit(
                X_val_meta[meta_fit_mask],
                y_res_val[meta_fit_mask],
            )

            oof_pred_full = clip_rating(
                y_val_base + model.predict(X_val_meta)
            )

            oof_select_res = evaluate_predictions(
                y_val_true[meta_select_mask],
                oof_pred_full[meta_select_mask],
                f"elasticnet_residual_meta_alpha_{alpha}_l1_{l1_ratio}",
            )

            row = {
                "alpha": float(alpha),
                "l1_ratio": float(l1_ratio),
                "meta_select_rmse": float(oof_select_res["rmse"]),
                "meta_select_mae": float(oof_select_res["mae"]),
            }
            elasticnet_rows.append(row)

            key = (
                float(oof_select_res["rmse"]),
                float(oof_select_res["mae"]),
            )

            if key < best_elasticnet_key:
                best_elasticnet_key = key
                best_elasticnet_cfg = {
                    "alpha": float(alpha),
                    "l1_ratio": float(l1_ratio),
                }

    if best_elasticnet_cfg is not None:
        elasticnet_final = ElasticNet(
            alpha=float(best_elasticnet_cfg["alpha"]),
            l1_ratio=float(best_elasticnet_cfg["l1_ratio"]),
            fit_intercept=True,
            max_iter=3000,
            tol=1e-3,
            random_state=42,
        )

        # refit selected config on full OOF for final test prediction
        elasticnet_final.fit(
            X_val_meta,
            y_res_val,
        )

        elasticnet_oof_pred = clip_rating(
            y_val_base + elasticnet_final.predict(X_val_meta)
        )

        elasticnet_test_pred = clip_rating(
            y_base + elasticnet_final.predict(X_test_meta)
        )

        elasticnet_weights = {
            "intercept": float(elasticnet_final.intercept_),
            "coef_by_feature": {
                name: float(coef)
                for name, coef in zip(meta_feature_names, elasticnet_final.coef_)
            },
        }

        elasticnet_params = {
            "alpha": float(best_elasticnet_cfg["alpha"]),
            "l1_ratio": float(best_elasticnet_cfg["l1_ratio"]),
            "fit_intercept": True,
            "feature_names": meta_feature_names,
            "selection_split": "meta_fit_70_meta_select_30",
        }

        register_candidate(
            "elasticnet_residual_meta",
            "meta_elasticnet",
            elasticnet_oof_pred,
            elasticnet_test_pred,
            weights=elasticnet_weights,
            params=elasticnet_params,
            eligible_for_selection=True,
        )

        advanced_diagnostics["candidate_search"]["elasticnet_residual_meta"] = {
            "grid_results": elasticnet_rows,
            "selected": elasticnet_params,
            "weights": elasticnet_weights,
        }

        print("Selected elasticnet_residual_meta:", elasticnet_params)

except Exception as exc:
    advanced_diagnostics["skipped_candidates"]["elasticnet_residual_meta"] = str(exc)
    print("Skipped elasticnet_residual_meta:", exc)


# -------------------------------------------------
# 8.6 Advanced 3 — HistGradientBoosting Residual Meta
# -------------------------------------------------
hgbr_rows = []
best_hgbr_cfg = None
best_hgbr_key = (float("inf"), float("inf"))

try:
    rng = np.random.RandomState(42)

    fit_indices = np.where(meta_fit_mask)[0]

    if len(fit_indices) > 300000:
        fit_indices = rng.choice(
            fit_indices,
            size=300000,
            replace=False,
        )

    X_train_hg = X_val_meta[fit_indices]
    y_train_hg = y_res_val[fit_indices]

    for max_iter in [100, 200]:
        for learning_rate in [0.03, 0.05]:
            for max_leaf_nodes in [15, 31]:
                for l2_regularization in [0.0, 0.1]:
                    model = HistGradientBoostingRegressor(
                        loss="squared_error",
                        max_iter=int(max_iter),
                        learning_rate=float(learning_rate),
                        max_leaf_nodes=int(max_leaf_nodes),
                        l2_regularization=float(l2_regularization),
                        random_state=42,
                    )

                    model.fit(
                        X_train_hg,
                        y_train_hg,
                    )

                    oof_pred_full = clip_rating(
                        y_val_base + model.predict(X_val_meta)
                    )

                    oof_select_res = evaluate_predictions(
                        y_val_true[meta_select_mask],
                        oof_pred_full[meta_select_mask],
                        (
                            "hist_gradient_boosting_residual_"
                            f"{max_iter}_{learning_rate}_{max_leaf_nodes}_{l2_regularization}"
                        ),
                    )

                    row = {
                        "max_iter": int(max_iter),
                        "learning_rate": float(learning_rate),
                        "max_leaf_nodes": int(max_leaf_nodes),
                        "l2_regularization": float(l2_regularization),
                        "train_rows_used": int(len(fit_indices)),
                        "meta_select_rmse": float(oof_select_res["rmse"]),
                        "meta_select_mae": float(oof_select_res["mae"]),
                    }
                    hgbr_rows.append(row)

                    key = (
                        float(oof_select_res["rmse"]),
                        float(oof_select_res["mae"]),
                    )

                    if key < best_hgbr_key:
                        best_hgbr_key = key
                        best_hgbr_cfg = {
                            "max_iter": int(max_iter),
                            "learning_rate": float(learning_rate),
                            "max_leaf_nodes": int(max_leaf_nodes),
                            "l2_regularization": float(l2_regularization),
                        }

    if best_hgbr_cfg is not None:
        final_indices = np.arange(len(X_val_meta))

        if len(final_indices) > 500000:
            final_indices = rng.choice(
                final_indices,
                size=300000,
                replace=False,
            )

        hgbr_final = HistGradientBoostingRegressor(
            loss="squared_error",
            max_iter=int(best_hgbr_cfg["max_iter"]),
            learning_rate=float(best_hgbr_cfg["learning_rate"]),
            max_leaf_nodes=int(best_hgbr_cfg["max_leaf_nodes"]),
            l2_regularization=float(best_hgbr_cfg["l2_regularization"]),
            random_state=42,
        )

        # refit selected config on full/sampled OOF for final test prediction
        hgbr_final.fit(
            X_val_meta[final_indices],
            y_res_val[final_indices],
        )

        hgbr_oof_pred = clip_rating(
            y_val_base + hgbr_final.predict(X_val_meta)
        )

        hgbr_test_pred = clip_rating(
            y_base + hgbr_final.predict(X_test_meta)
        )

        hgbr_params = {
            "max_iter": int(best_hgbr_cfg["max_iter"]),
            "learning_rate": float(best_hgbr_cfg["learning_rate"]),
            "max_leaf_nodes": int(best_hgbr_cfg["max_leaf_nodes"]),
            "l2_regularization": float(best_hgbr_cfg["l2_regularization"]),
            "train_rows_used_meta_fit": int(len(fit_indices)),
            "train_rows_used_final": int(len(final_indices)),
            "feature_names": meta_feature_names,
            "selection_split": "meta_fit_70_meta_select_30",
        }

        register_candidate(
            "hist_gradient_boosting_residual",
            "meta_hist_gradient_boosting",
            hgbr_oof_pred,
            hgbr_test_pred,
            weights={"model_type": "HistGradientBoostingRegressor"},
            params=hgbr_params,
            eligible_for_selection=True,
        )

        advanced_diagnostics["candidate_search"]["hist_gradient_boosting_residual"] = {
            "grid_results": hgbr_rows,
            "selected": hgbr_params,
            "weights": {"model_type": "HistGradientBoostingRegressor"},
        }

        print("Selected hist_gradient_boosting_residual:", hgbr_params)

except Exception as exc:
    advanced_diagnostics["skipped_candidates"]["hist_gradient_boosting_residual"] = str(exc)
    print("Skipped hist_gradient_boosting_residual:", exc)


# -------------------------------------------------
# 8.7 Advanced 4 — Group-aware Ridge
# -------------------------------------------------
user_count_map_oof = train_fit_stage4.groupby(COL_USER).size()
item_count_map_oof = train_fit_stage4.groupby(COL_ITEM).size()

user_count_map_test = train.groupby(COL_USER).size()
item_count_map_test = train.groupby(COL_ITEM).size()

user_count_val = (
    validation_index_oof[COL_USER]
    .map(user_count_map_oof)
    .fillna(0)
    .astype(np.int32)
    .values
)

item_count_val = (
    validation_index_oof[COL_ITEM]
    .map(item_count_map_oof)
    .fillna(0)
    .astype(np.int32)
    .values
)

user_count_test = (
    test_warm[COL_USER]
    .map(user_count_map_test)
    .fillna(0)
    .astype(np.int32)
    .values
)

item_count_test = (
    test_warm[COL_ITEM]
    .map(item_count_map_test)
    .fillna(0)
    .astype(np.int32)
    .values
)

val_groups = build_group_labels(
    user_count_val,
    item_count_val,
)

test_groups = build_group_labels(
    user_count_test,
    item_count_test,
)

group_oof_pred, group_test_pred, group_summary = fit_group_aware_ridge_candidate(
    X_val_base_b_c_d,
    X_test_base_b_c_d,
    y_val_true,
    y_pred_ridge_oof,
    y_pred_ridge,
    val_groups,
    test_groups,
    min_rows=5000,
    alpha=1.0,
)

# Not eligible because current group-aware implementation fits/evaluates within same OOF groups.
register_candidate(
    "group_aware_ridge",
    "group_aware_ridge",
    group_oof_pred,
    group_test_pred,
    weights={
        "fallback_model": "ridge_positive_base_b_c_d",
    },
    params={
        "alpha": 1.0,
        "positive": True,
        "fit_intercept": False,
        "feature_names": ["base", "b", "c", "d"],
        "min_group_rows": 5000,
        "group_summary": group_summary,
        "note": (
            "not eligible for selection because group-specific models "
            "fit/evaluate on same OOF groups"
        ),
    },
    eligible_for_selection=False,
)

advanced_diagnostics["candidate_search"]["group_aware_ridge"] = {
    "feature_names": ["base", "b", "c", "d"],
    "min_group_rows": 5000,
    "group_summary": group_summary,
    "eligible_for_selection": False,
}


# -------------------------------------------------
# 8.8 Display comparison
# -------------------------------------------------
ensemble_extra_df = (
    pd.DataFrame(compare_rows)
    .sort_values(
        ["eligible_for_selection", "oof_rmse", "oof_mae", "model"],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

display(ensemble_extra_df)

print("=" * 70)
print("CELL 8 DONE")
print("=" * 70)

CELL 8 — RESIDUAL BLEND + GLOBAL RIDGE + ADVANCED META
Selected residual_ridge_meta: {'alpha': 0.1, 'fit_intercept': True, 'feature_names': ['b_minus_base', 'c_minus_base', 'd_minus_base', 'a_minus_base', 'abs_b_minus_base', 'abs_c_minus_base', 'abs_d_minus_base', 'abs_a_minus_base', 'prediction_mean', 'prediction_std', 'prediction_range', 'base'], 'selection_split': 'meta_fit_70_meta_select_30'}
Selected elasticnet_residual_meta: {'alpha': 0.0001, 'l1_ratio': 0.5, 'fit_intercept': True, 'feature_names': ['b_minus_base', 'c_minus_base', 'd_minus_base', 'a_minus_base', 'abs_b_minus_base', 'abs_c_minus_base', 'abs_d_minus_base', 'abs_a_minus_base', 'prediction_mean', 'prediction_std', 'prediction_range', 'base'], 'selection_split': 'meta_fit_70_meta_select_30'}
Selected hist_gradient_boosting_residual: {'max_iter': 100, 'learning_rate': 0.05, 'max_leaf_nodes': 15, 'l2_regularization': 0.1, 'train_rows_used_meta_fit': 79181, 'train_rows_used_final': 113117, 'feature_names': ['b_minus_base

,model,family,eligible_for_selection,oof_rmse,oof_mae,oof_full_rmse,oof_full_mae,test_rmse,test_mae,weights_json,params_json,selection_source
0,hist_gradient_boosting_residual,meta_hist_gradient_boosting,True,1.043145,0.747794,1.021772,0.735922,1.093688,0.805933,"{""model_type"": ""HistGradientBoostingRegressor""}","{""feature_names"": [""b_minus_base"", ""c_minus_ba...",meta_select_from_oof_validation
1,model_d,single_model,True,1.046053,0.743093,1.046053,0.743093,1.102868,0.811267,"{""d"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
2,baseline,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""base"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
3,model_c,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""c"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
4,model_b,single_model,True,1.046745,0.726074,1.046745,0.726074,1.108873,0.764589,"{""b"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
5,residual_ridge_meta,meta_ridge,True,1.047929,0.754647,1.027351,0.742900,1.189166,0.834786,"{""coef_by_feature"": {""a_minus_base"": 0.1559196...","{""alpha"": 0.1, ""feature_names"": [""b_minus_base...",meta_select_from_oof_validation
6,elasticnet_residual_meta,meta_elasticnet,True,1.047959,0.754704,1.027378,0.742966,1.186653,0.833484,"{""coef_by_feature"": {""a_minus_base"": 0.2618804...","{""alpha"": 0.0001, ""feature_names"": [""b_minus_b...",meta_select_from_oof_validation
7,model_a,single_model,True,1.048409,0.742963,1.048409,0.742963,1.105582,0.810892,"{""a"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
8,grid_base_b_c,grid_blend,True,1.065895,0.746736,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.15000000000000002, ""c"": 0...","{""keys"": [""base"", ""b"", ""c""], ""selection_source...",meta_select_from_oof_validation
9,grid_base_b_c_d,grid_blend,True,1.065895,0.746736,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.2, ""c"": 0.300000000000000...","{""keys"": [""base"", ""b"", ""c"", ""d""], ""selection_s...",meta_select_from_oof_validation


CELL 8 DONE


In [9]:
# ============================================
# CELL 9 — COMPARE AND SELECT BY VALIDATION ONLY
# ============================================

compare_df = (
    pd.DataFrame(compare_rows)
    .sort_values(
        ["eligible_for_selection", "oof_rmse", "oof_mae", "model"],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

display(compare_df)

eligible_df = (
    compare_df[compare_df["eligible_for_selection"]]
    .sort_values(["oof_rmse", "oof_mae", "model"])
    .reset_index(drop=True)
)

assert len(eligible_df) > 0, "No eligible candidates were generated."

best_stage4_name = eligible_df.iloc[0]["model"]
best_stage4_candidate = candidate_store[best_stage4_name]

best_stage4_family = best_stage4_candidate["family"]
best_stage4_pred = clip_rating(best_stage4_candidate["test_pred"]).astype(np.float32)
best_stage4_oof_pred = clip_rating(best_stage4_candidate["oof_pred"]).astype(np.float32)
best_stage4_weights = best_stage4_candidate["weights"]
best_stage4_params = best_stage4_candidate["params"]

best_stage4_oof_metrics_full = evaluate_predictions(
    y_val_true,
    best_stage4_oof_pred,
    best_stage4_name,
)

best_stage4_oof_metrics_select = evaluate_predictions(
    y_val_true[meta_select_mask],
    best_stage4_oof_pred[meta_select_mask],
    best_stage4_name,
)

best_stage4_test_metrics = evaluate_predictions(
    y_true,
    best_stage4_pred,
    best_stage4_name,
)

baseline_test_metrics = evaluate_predictions(y_true, y_base, "baseline")
baseline_test_per_star = evaluate_per_star(y_true, y_base)
best_test_per_star = evaluate_per_star(y_true, best_stage4_pred)

comparison_vs_current_ridge = None
if "ridge_positive_base_b_c_d" in candidate_store:
    ridge_candidate = candidate_store["ridge_positive_base_b_c_d"]

    ridge_oof_metrics_full = evaluate_predictions(
        y_val_true,
        ridge_candidate["oof_pred"],
        "ridge_positive_base_b_c_d",
    )

    ridge_oof_metrics_select = evaluate_predictions(
        y_val_true[meta_select_mask],
        ridge_candidate["oof_pred"][meta_select_mask],
        "ridge_positive_base_b_c_d",
    )

    ridge_test_metrics = evaluate_predictions(
        y_true,
        ridge_candidate["test_pred"],
        "ridge_positive_base_b_c_d",
    )

    comparison_vs_current_ridge = {
        "reference_model": "ridge_positive_base_b_c_d",
        "reference_family": ridge_candidate["family"],
        "oof_reference_metrics_full": ridge_oof_metrics_full,
        "oof_reference_metrics_select": ridge_oof_metrics_select,
        "test_reference_metrics": ridge_test_metrics,
        "oof_select_delta_rmse": float(best_stage4_oof_metrics_select["rmse"] - ridge_oof_metrics_select["rmse"]),
        "oof_select_delta_mae": float(best_stage4_oof_metrics_select["mae"] - ridge_oof_metrics_select["mae"]),
        "test_delta_rmse": float(best_stage4_test_metrics["rmse"] - ridge_test_metrics["rmse"]),
        "test_delta_mae": float(best_stage4_test_metrics["mae"] - ridge_test_metrics["mae"]),
    }

advanced_ensemble_improved = bool(
    best_stage4_family in ADVANCED_FAMILIES
    and comparison_vs_current_ridge is not None
    and (
        comparison_vs_current_ridge["test_delta_rmse"] < 0
        or comparison_vs_current_ridge["test_delta_mae"] < 0
    )
)

selected_preview = {
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "selected_by": "validation_rmse_then_validation_mae",
    "selection_source": "oof_validation_or_meta_select",
    "oof_metrics_full": to_jsonable(best_stage4_oof_metrics_full),
    "oof_metrics_select": to_jsonable(best_stage4_oof_metrics_select),
    "test_metrics": to_jsonable(best_stage4_test_metrics),
    "weights": to_jsonable(best_stage4_weights),
    "params": to_jsonable(best_stage4_params),
    "note": "test_warm was used only for final evaluation, not selection",
}

print("Best validation candidate:", best_stage4_name)
print(json.dumps(to_jsonable(selected_preview), indent=2))

print("Best test result after applying selected candidate:")
print(json.dumps(to_jsonable(best_stage4_test_metrics), indent=2))

if comparison_vs_current_ridge is not None:
    print("Comparison vs ridge_positive_base_b_c_d:")
    print(json.dumps(to_jsonable(comparison_vs_current_ridge), indent=2))
    print("Advanced ensemble improved over ridge_positive_base_b_c_d:", advanced_ensemble_improved)

,model,family,eligible_for_selection,oof_rmse,oof_mae,oof_full_rmse,oof_full_mae,test_rmse,test_mae,weights_json,params_json,selection_source
0,hist_gradient_boosting_residual,meta_hist_gradient_boosting,True,1.043145,0.747794,1.021772,0.735922,1.093688,0.805933,"{""model_type"": ""HistGradientBoostingRegressor""}","{""feature_names"": [""b_minus_base"", ""c_minus_ba...",meta_select_from_oof_validation
1,model_d,single_model,True,1.046053,0.743093,1.046053,0.743093,1.102868,0.811267,"{""d"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
2,baseline,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""base"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
3,model_c,single_model,True,1.046057,0.743151,1.046057,0.743151,1.102864,0.811304,"{""c"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
4,model_b,single_model,True,1.046745,0.726074,1.046745,0.726074,1.108873,0.764589,"{""b"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
5,residual_ridge_meta,meta_ridge,True,1.047929,0.754647,1.027351,0.742900,1.189166,0.834786,"{""coef_by_feature"": {""a_minus_base"": 0.1559196...","{""alpha"": 0.1, ""feature_names"": [""b_minus_base...",meta_select_from_oof_validation
6,elasticnet_residual_meta,meta_elasticnet,True,1.047959,0.754704,1.027378,0.742966,1.186653,0.833484,"{""coef_by_feature"": {""a_minus_base"": 0.2618804...","{""alpha"": 0.0001, ""feature_names"": [""b_minus_b...",meta_select_from_oof_validation
7,model_a,single_model,True,1.048409,0.742963,1.048409,0.742963,1.105582,0.810892,"{""a"": 1.0}","{""kind"": ""single_model""}",full_oof_validation
8,grid_base_b_c,grid_blend,True,1.065895,0.746736,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.15000000000000002, ""c"": 0...","{""keys"": [""base"", ""b"", ""c""], ""selection_source...",meta_select_from_oof_validation
9,grid_base_b_c_d,grid_blend,True,1.065895,0.746736,1.045229,0.734311,1.094008,0.784323,"{""b"": 0.5, ""base"": 0.2, ""c"": 0.300000000000000...","{""keys"": [""base"", ""b"", ""c"", ""d""], ""selection_s...",meta_select_from_oof_validation


Best validation candidate: hist_gradient_boosting_residual
{
  "selected_model": "hist_gradient_boosting_residual",
  "selected_family": "meta_hist_gradient_boosting",
  "selected_by": "validation_rmse_then_validation_mae",
  "selection_source": "oof_validation_or_meta_select",
  "oof_metrics_full": {
    "model": "hist_gradient_boosting_residual",
    "n": 113117,
    "rmse": 1.0217721216519882,
    "mae": 0.7359215617179871
  },
  "oof_metrics_select": {
    "model": "hist_gradient_boosting_residual",
    "n": 33936,
    "rmse": 1.043145268393614,
    "mae": 0.7477944493293762
  },
  "test_metrics": {
    "model": "hist_gradient_boosting_residual",
    "n": 240285,
    "rmse": 1.0936880366620916,
    "mae": 0.8059325814247131
  },
  "weights": {
    "model_type": "HistGradientBoostingRegressor"
  },
  "params": {
    "max_iter": 100,
    "learning_rate": 0.05,
    "max_leaf_nodes": 15,
    "l2_regularization": 0.1,
    "train_rows_used_meta_fit": 79181,
    "train_rows_used_final": 1

In [10]:
# ============================================
# CELL 10 — SAVE STAGE 4 OUTPUTS
# ============================================

stage4_best_result = {
    "label": "best validation-selected ensemble so far",
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "selected_by": "validation_rmse_then_validation_mae",
    "selection_source": "oof_validation_or_meta_select",
    "oof_metrics_full": to_jsonable(best_stage4_oof_metrics_full),
    "oof_metrics_select": to_jsonable(best_stage4_oof_metrics_select),
    "test_metrics": to_jsonable(best_stage4_test_metrics),
    "weights": to_jsonable(best_stage4_weights),
    "params": to_jsonable(best_stage4_params),
    "note": "test_warm was used only for final evaluation, not selection",
}

stage4_weights = {
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "selected_by": "validation_rmse_then_validation_mae",
    "selected_weights": to_jsonable(best_stage4_weights),
    "selected_params": to_jsonable(best_stage4_params),
    "all_candidate_weights": {
        name: to_jsonable(info["weights"])
        for name, info in candidate_store.items()
    },
    "all_candidate_params": {
        name: to_jsonable(info["params"])
        for name, info in candidate_store.items()
    },
    "selection_source": "oof_validation_or_meta_select",
}

stage4_advanced_diagnostics = {
    "selected_model": best_stage4_name,
    "selected_family": best_stage4_family,
    "best_oof_metrics_full": to_jsonable(best_stage4_oof_metrics_full),
    "best_oof_metrics_select": to_jsonable(best_stage4_oof_metrics_select),
    "best_test_metrics": to_jsonable(best_stage4_test_metrics),
    "baseline_test_metrics": to_jsonable(baseline_test_metrics),
    "best_test_per_star": to_jsonable(best_test_per_star),
    "baseline_test_per_star": to_jsonable(baseline_test_per_star),
    "comparison_vs_current_ridge_positive_base_b_c_d": to_jsonable(comparison_vs_current_ridge),
    "advanced_ensemble_improved_over_current_ridge": bool(advanced_ensemble_improved),
    "candidate_search": to_jsonable(advanced_diagnostics["candidate_search"]),
    "skipped_candidates": to_jsonable(advanced_diagnostics["skipped_candidates"]),
    "meta_feature_names": to_jsonable(advanced_diagnostics["meta_feature_names"]),
    "meta_split": {
        "fit_rows": int(meta_fit_mask.sum()),
        "select_rows": int(meta_select_mask.sum()),
        "fit_ratio": 0.70,
    },
}

np.save(STAGE4_DIR / "stage4_best_preds.npy", best_stage4_pred)
compare_df.to_csv(STAGE4_DIR / "stage4_compare.csv", index=False)

with open(STAGE4_DIR / "stage4_best_result.json", "w") as f:
    json.dump(to_jsonable(stage4_best_result), f, indent=2)

with open(STAGE4_DIR / "stage4_weights.json", "w") as f:
    json.dump(to_jsonable(stage4_weights), f, indent=2)

with open(STAGE4_DIR / "stage4_advanced_diagnostics.json", "w") as f:
    json.dump(to_jsonable(stage4_advanced_diagnostics), f, indent=2)

print("Saved:")
print(STAGE4_DIR / "stage4_best_result.json")
print(STAGE4_DIR / "stage4_best_preds.npy")
print(STAGE4_DIR / "stage4_compare.csv")
print(STAGE4_DIR / "stage4_weights.json")
print(STAGE4_DIR / "stage4_advanced_diagnostics.json")

Saved:
/kaggle/working/stage4_artifacts/stage4_best_result.json
/kaggle/working/stage4_artifacts/stage4_best_preds.npy
/kaggle/working/stage4_artifacts/stage4_compare.csv
/kaggle/working/stage4_artifacts/stage4_weights.json
/kaggle/working/stage4_artifacts/stage4_advanced_diagnostics.json


The Stage 4 notebook selects its final candidate only by strict OOF validation from Stage 3.

Advanced candidates now include residual ridge meta, elasticnet residual meta, HistGradientBoosting residual meta, and group-aware ridge. `test_warm` is applied only once after OOF selection to report final metrics.

If the selected candidate improves RMSE but worsens MAE versus `ridge_positive_base_b_c_d` or `model_b`, treat that as an explicit trade-off rather than a best-overall claim.
